In [0]:
CREATE TABLE IF NOT EXISTS workspace.default.datasetS8 AS
SELECT * FROM workspace.ucm.bronze_retail_transactions_course_dataset;

In [0]:
CREATE OR REPLACE TABLE workspace.default.datasets8_part
USING DELTA
PARTITIONED BY (year_month)
AS
SELECT
  *,
  date_format(transaction_ts, 'yyyy-MM') AS year_month
FROM workspace.default.datasetS8;

In [0]:
SHOW PARTITIONS workspace.default.datasets8_part;

In [0]:
SELECT year_month, COUNT(*) AS row_count
FROM workspace.default.datasets8_part
GROUP BY year_month
ORDER BY year_month;

In [0]:
SELECT *
FROM workspace.default.datasets8_part
WHERE year_month = '2024-03';

In [0]:
SELECT *
FROM workspace.default.datasets8_part
WHERE store_id = 'S001';

In [0]:
SHOW PARTITIONS datasets8_part;

In [0]:
SELECT * FROM workspace.default.datasets8_part LIMIT 5;

SHOW PARTITIONS workspace.default.datasets8_part;

In [0]:
CREATE OR REPLACE TABLE workspace.default.datasets8_big
USING DELTA
PARTITIONED BY (year_month)
AS
SELECT t.*, r.id AS dup_id
FROM workspace.default.datasets8_part t
CROSS JOIN range(0, 200) r;

In [0]:
SELECT COUNT(*) AS rows
FROM workspace.default.datasets8_big;

SHOW PARTITIONS workspace.default.datasets8_big;

In [0]:
SELECT *
FROM workspace.default.datasets8_big
WHERE year_month = '2024-03'
  AND customer_id = 'C00190'
  AND product_id = 'P1001';

In [0]:
OPTIMIZE workspace.default.datasets8_big
ZORDER BY (customer_id, product_id);

In [0]:
SELECT *
FROM workspace.default.datasets8_big
WHERE year_month = '2024-03'
  AND customer_id = 'C00190'
  AND product_id = 'P1001';

In [0]:
-- Before/After results should match
SELECT COUNT(*) AS matched_rows
FROM workspace.default.datasets8_big
WHERE year_month = '2024-03'
  AND customer_id = 'C00190'
  AND product_id = 'P1001';

In [0]:
-- Z-ORDER helps when filtering by multiple frequently-used columns
SELECT *
FROM workspace.default.datasets8_big
WHERE customer_id = 'C00190'
  AND product_id  = 'P1001'
  AND store_id    = 'S001';

In [0]:
%python
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import mlflow
import mlflow.spark

       
from pyspark.sql import functions as F



In [0]:
%python
from pyspark.sql import functions as F

# Load base table
df = spark.table("workspace.default.datasets8")

# Create total_value and label
df_hv = (
    df
    .withColumn("total_value", F.col("quantity") * F.col("unit_price_eur"))
    .withColumn("high_value_flag", F.when(F.col("total_value") > 50, 1).otherwise(0))
)

# Quick check
df_hv.groupBy("high_value_flag").count().show()

In [0]:
%python
df_model = df_hv.select(
    "store_id",
    "product_id",
    "customer_id",
    "quantity",
    "discount_pct",
    "shipping_days",
    "high_value_flag"
)

df_model.printSchema()
df_model.show(5)

In [0]:
%python
df_clean = (
    df_model
    # numeric cleanup (turn bad strings into NULL safely)
    .withColumn("discount_pct", F.regexp_replace("discount_pct", ",", "."))
    .withColumn("discount_pct", F.when(F.col("discount_pct") == "N/A", None).otherwise(F.col("discount_pct")).cast("double"))
    .withColumn("shipping_days", F.col("shipping_days").cast("double"))
    .withColumn("quantity", F.col("quantity").cast("double"))
    .withColumn("label", F.col("high_value_flag").cast("double"))
    .drop("high_value_flag")
)

# Drop rows with missing values in any required column
df_clean = df_clean.dropna(subset=["store_id","product_id","customer_id","quantity","discount_pct","shipping_days","label"])

df_clean.groupBy("label").count().show()

In [0]:
%python
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Convert strings -> numbers
idx_store = StringIndexer(inputCol="store_id", outputCol="store_id_idx", handleInvalid="keep")
idx_product = StringIndexer(inputCol="product_id", outputCol="product_id_idx", handleInvalid="keep")
idx_customer = StringIndexer(inputCol="customer_id", outputCol="customer_id_idx", handleInvalid="keep")

df_prepared = (
    idx_store.fit(df_clean).transform(df_clean)
)
df_prepared = idx_product.fit(df_prepared).transform(df_prepared)
df_prepared = idx_customer.fit(df_prepared).transform(df_prepared)

# Assemble into "features"
assembler = VectorAssembler(
    inputCols=["store_id_idx","product_id_idx","customer_id_idx","quantity","discount_pct","shipping_days"],
    outputCol="features"
)

ml_df = assembler.transform(df_prepared).select("features", "label")
ml_df.show(5, truncate=False)

In [0]:
%python
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

print("Train distribution:")
train_df.groupBy("label").count().show()

print("Test distribution:")
test_df.groupBy("label").count().show()

In [0]:
%python
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
model = lr.fit(train_df)

print("Model trained.")

In [0]:
%python
predictions = model.transform(test_df)

predictions.select("label", "prediction", "probability").show(10, truncate=False)

In [0]:
%python
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(predictions)
print("AUC:", auc)

# Confusion-style counts
predictions.groupBy("label", "prediction").count().orderBy("label","prediction").show()

In [0]:
%python
# Stage 4.1 — Experiment Tracking with MLflow
import mlflow
import mlflow.spark

# (Optional) give your experiment a name so runs are grouped nicely
mlflow.set_experiment("/Shared/datasets8_high_value")

with mlflow.start_run(run_name="HV_LogReg_v1"):
    # Log parameters (the settings we chose)
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("train_rows", train_df.count())
    mlflow.log_param("test_rows", test_df.count())

    # Log the evaluation metric we already computed (AUC)
    mlflow.log_metric("AUC", float(auc))

    print("✅ MLflow run created and metrics logged.")
    print("Run ID:", mlflow.active_run().info.run_id)

In [0]:
CREATE VOLUME IF NOT EXISTS workspace.default.mlflow_tmp;

In [0]:
%python
import os

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlflow_tmp"
print("MLFLOW_DFS_TMP set to:", os.environ["MLFLOW_DFS_TMP"])

In [0]:
%python
# HV Stage 5 — Model Versioning (Register + keep metrics/params)

import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature

model_name = "High_Value_Predictor"

# (optional) keep the same experiment
mlflow.set_experiment("/Shared/datasets8_high_value")

# Create a small input example for the model signature
input_example_pd = test_df.limit(5).toPandas()
preds_example_pd = model.transform(test_df.limit(5)).toPandas()
signature = infer_signature(input_example_pd, preds_example_pd)

with mlflow.start_run(run_name="HV_LogReg_v1_register"):
    # Log the same params/metrics so this run looks complete in the UI
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("train_rows", train_df.count())
    mlflow.log_param("test_rows", test_df.count())
    mlflow.log_metric("AUC", float(auc))

    # Log + register the model
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="model",
        registered_model_name=model_name,
        signature=signature
    )

print("✅ Model registered (with params + metrics).")


In [0]:
%python
import mlflow.spark

# IMPORTANT: use the exact registered name you see in the UI
# That is:  workspace.default.high_value_predictor (v1)
model_uri = "models:/workspace.default.high_value_predictor/1"

loaded_spark_model = mlflow.spark.load_model(model_uri)

preds = loaded_spark_model.transform(test_df)

preds.select("label", "prediction", "probability").show(10, truncate=False)

